# Segmentation Walkthrough

Phase 3 of the platform: **semantic**, **instance**, and **promptable** segmentation,
all sharing the Phase 1 core (config, registry, Trainer, evaluators).

- **Semantic** (`unet`, `deeplabv3plus`, `smp`, `segformer`) — `model(images) -> logits (B, C, H, W)`,
  trained through the base `Trainer` with a pixel-wise criterion (CE / Dice / CE+Dice).
- **Instance** (`mask_rcnn`) — extends Phase 2 Faster R-CNN with a mask branch; reuses the
  detection stack and reports mask (segm) mAP.
- **Promptable / universal** (`sam`, `mask2former`, `oneformer`) — HuggingFace wrappers.

Everything here runs offline on CPU against the synthetic shapes fixtures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)

## 1. The synthetic shapes-with-masks dataset

`synthetic_shapes_seg` is the semantic twin of the Phase 2 `synthetic_shapes` detection
fixture — *the same images*, with the drawn pixels as a class-index mask
(0 = background, 1 = rectangle, 2 = circle, 3 = triangle).

In [ ]:
from image_analytics.data.datasets import build_dataset
from image_analytics.core.config import DataConfig

ds = build_dataset(DataConfig(dataset='synthetic_shapes_seg',
                              kwargs={'size': 64, 'image_size': 96}), split='train')
print('classes:', ds.CLASSES, '| num_classes:', ds.num_classes, '| samples:', len(ds))

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for col in range(4):
    image, mask = ds[col]
    axes[0, col].imshow(image.permute(1, 2, 0))
    axes[0, col].set_title(f'image {col}'); axes[0, col].axis('off')
    axes[1, col].imshow(mask, vmin=0, vmax=3, cmap='tab10')
    axes[1, col].set_title('mask'); axes[1, col].axis('off')
plt.tight_layout(); plt.show()

## 2. Train U-Net from scratch

U-Net wraps any registered pyramid backbone as the encoder. We load the checked-in config
and override a few keys for a short CPU run.

In [ ]:
from image_analytics.core.config import load_config
from image_analytics.segmentation.train import run

config = load_config('../configs/segmentation/unet_shapes.yaml', overrides=[
    'training.epochs=5', 'training.device=cpu', 'training.warmup_epochs=0',
    'data.kwargs.size=192', 'model.backbone.pretrained=false', 'output_dir=outputs',
])
metrics = run(config)
print({k: round(v, 4) for k, v in metrics.items() if '/' in k and 'iou_' not in k})

## 3. Visualize U-Net predictions

Load the best checkpoint and argmax the logits into a predicted class map.

In [ ]:
from image_analytics.segmentation.train import build_segmentation_model
from image_analytics.data.transforms.segmentation import build_segmentation_transforms

model = build_segmentation_model(config.model).eval()
state = torch.load('outputs/unet_shapes/checkpoints/best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state['model'])

val = build_dataset(DataConfig(dataset='synthetic_shapes_seg', kwargs={'size': 8, 'image_size': 96}),
                    split='val', transform=build_segmentation_transforms(96, train=False))
raw = build_dataset(DataConfig(dataset='synthetic_shapes_seg', kwargs={'size': 8, 'image_size': 96}), split='val')

fig, axes = plt.subplots(3, 4, figsize=(11, 8))
with torch.no_grad():
    for col in range(4):
        image, _ = val[col]
        pred = model(image.unsqueeze(0)).argmax(1)[0]
        axes[0, col].imshow(raw[col][0].permute(1, 2, 0)); axes[0, col].set_title('image'); axes[0, col].axis('off')
        axes[1, col].imshow(raw[col][1], vmin=0, vmax=3, cmap='tab10'); axes[1, col].set_title('ground truth'); axes[1, col].axis('off')
        axes[2, col].imshow(pred, vmin=0, vmax=3, cmap='tab10'); axes[2, col].set_title('prediction'); axes[2, col].axis('off')
plt.tight_layout(); plt.show()

## 4. DeepLabv3+ and the smp library

`deeplabv3plus` (from scratch, ASPP + low-level fusion) swaps in by changing `model.name`;
the pipeline sets `output_stride=16` and the right encoder levels automatically:

```bash
python scripts/train.py --config configs/segmentation/deeplabv3plus_shapes.yaml
```

The `smp` wrapper (needs `pip install 'image-analytics[seg]'`) adds U-Net++, MAnet, Linknet,
PSPNet, PAN … over 400+ timm encoders behind the same interface — e.g.
`model.name: smp`, `model.kwargs: {arch: unetplusplus, encoder_name: tu-convnext_tiny}`.

## 5. SegFormer fine-tuning (HuggingFace)

`segformer` upsamples HF's 1/4-resolution logits, so it trains through the same base
Trainer + criterion as the from-scratch models. Pretrained weights download on first run:

```bash
python scripts/train.py --config configs/segmentation/segformer_shapes.yaml
```

In [ ]:
# Random-init SegFormer forward (offline, tiny config) — no weights download.
from image_analytics.core.registry import MODELS
import image_analytics.segmentation.semantic  # registers segformer

tiny = dict(hidden_sizes=[8, 16, 32, 64], depths=[1, 1, 1, 1],
            num_attention_heads=[1, 1, 2, 4], decoder_hidden_size=32)
seg = MODELS.build('segformer', num_classes=4, pretrained=False, config_kwargs=tiny).eval()
with torch.no_grad():
    logits = seg(torch.randn(1, 3, 96, 96))
print('SegFormer logits:', tuple(logits.shape))

## 6. Instance segmentation — Mask R-CNN

`mask_rcnn` extends Phase 2 Faster R-CNN with a 14×14 RoIAlign mask branch (→ 28×28
per-class masks). It reuses the detection collate / `DetectionTrainer` and is evaluated
with mask (segm) mAP:

```bash
python scripts/train.py --config configs/segmentation/mask_rcnn_shapes.yaml
```

Below we just build a (random-init) model and visualize its predicted instance masks.

In [ ]:
from image_analytics.core.config import BackboneConfig, ModelConfig

mrcnn = build_segmentation_model(ModelConfig(
    name='mask_rcnn', num_classes=3,
    backbone=BackboneConfig(name='resnet18', pretrained=False, features_only=True),
    kwargs=dict(fpn_channels=64, box_head_dim=128, mask_head_dim=64,
                rpn_anchor_sizes=[[16], [32], [64], [128], [256]], rpn_post_nms_topk=[200, 100]),
)).eval()

inst = build_dataset(DataConfig(dataset='synthetic_shapes_instance',
                                kwargs={'size': 4, 'image_size': 96}), split='val')
image, target = inst[0]
with torch.no_grad():
    pred = mrcnn(image.unsqueeze(0))[0]
print('detections:', len(pred['boxes']), '| mask tensor:', tuple(pred['masks'].shape))
print('(random-init model — train with the config above for meaningful masks)')

## 7. Promptable & universal segmentation (HuggingFace)

All inference-only, weights download on first use (`pip install 'image-analytics[seg]'`):

```python
from image_analytics.core.registry import MODELS
import image_analytics.foundation, image_analytics.segmentation.instance, image_analytics.segmentation.panoptic

# SAM v1 — promptable (points / boxes). Runs on torch 2.2; SAM 2 needs torch>=2.3.1.
sam = MODELS.build('sam', model_name='facebook/sam-vit-base')
masks, scores = sam.predict(pil_image, points=[[[450, 600]]], labels=[[1]])

# Mask2Former / OneFormer — universal (semantic / instance / panoptic).
m2f = MODELS.build('mask2former', task='panoptic')
result = m2f.predict(pil_image)            # {'segmentation', 'segments_info'}
```

Panoptic results are scored with `PanopticQualityEvaluator` (PQ / SQ / RQ).

## Summary

| Task | Models | Trainer | Metric |
|---|---|---|---|
| Semantic | `unet`, `deeplabv3plus`, `smp`, `segformer` | base `Trainer` + criterion | mIoU / Dice |
| Instance | `mask_rcnn` | `DetectionTrainer` | mask mAP |
| Panoptic | `mask2former`, `oneformer` | — (inference) | PQ / SQ / RQ |
| Promptable | `sam` | — (inference) | — |

Next: Phase 4 (satellite / multi-spectral) reuses the U-Net decoder for change detection.